# COLUMN FILTERING NOTEBOOK

This notebook is used to extract the relevant columns needed for creating HCP Longitudinal Data, can edit the first cell if any extra columns are required in future.

List of columns that will be required further along

In [0]:
column_config = {

                    # EMAIL DATA
                    "individualemailresult_raw": [
                                                    "id",
                                                    "Person_Account_Id__c",
                                                    "et4ae5__TriggeredSendDefinition__c",
                                                    "et4ae5__Clicked__c",
                                                    "et4ae5__DateSent__c",
                                                    "et4ae5__FromName__c"
                                                ],

                    # ACCOUNT TERRITORIES (used in many joins)
                    "dw_tableau_dbo_accountallterritories_raw": [
                                                                    "AccountID",
                                                                    "IsActive",
                                                                    "IsPersonAccount",
                                                                    "EPLstatusPAH",
                                                                    "NPI",
                                                                    "npispecialty"
                                                                ],

                    # REFERRALS
                    "dw_processeddata_dbo_patientactivityhistorydetailed_raw": [
                                                                                    "PatientSequence",
                                                                                    "PatientID",
                                                                                    "ReferralPrescriberAccountID",
                                                                                    "ReferralDate",
                                                                                    "IsValidReferral",
                                                                                    "Brand"
                                                                                ],

                    # SPEAKER PROGRAM
                    "dw_datasets_centris_speakerprogram_raw": [
                                                                "ProgramID",
                                                                "ProgramStatus",
                                                                "ProgramStartDateTime",
                                                                "ProgramType"
                                                            ],

                    # SPEAKER PROGRAM ATTENDEE
                    "dw_datasets_centris_speakerprogramattendee_raw": [
                                                                            "ProgramID",
                                                                            "AttendanceID",
                                                                            "UTCustomerID",
                                                                            "AttendeeRole",
                                                                            "attendanceflag"
                                                                        ],

                    # SPEAKER PROGRAM BRAND TOPIC
                    "dw_datasets_centris_speakerprogrambrandtopic_raw": [
                                                                            "ProgramID",
                                                                            "ProductName"
                                                                        ],

                    # CALLS
                    "dw_veeva_dbo_call_raw": [
                                                "id",
                                                "Account_vod__c",
                                                "Call_Date_vod__c",
                                                "Call_Datetime_vod__c",
                                                "ownerid",
                                                "Status_vod__c",
                                                "Interaction_Type__c",
                                                "Subject_vod__c",
                                                "Scientific_Exchange__c",
                                                "CreatedDate"
                                            ],

                    # CALL DETAILS
                    "dw_veeva_dbo_calldetail_raw": [
                                                        "Call2_vod__c",
                                                        "Product_vod__c",
                                                        "Detail_Priority_vod__c"
                                                    ],

                    # PRODUCT
                    "dw_veeva_dbo_product_raw": [
                                                    "ID",
                                                    "Name"
                                                ],

                    # USERS
                    "users_raw": [
                                    "X18_Digit_User_ID__c",
                                    "Name",
                                    "email",
                                    "ProfileID"
                                ],

                    # PROFILE
                    "profile_raw": [
                                        "Id",
                                        "Name"
                                    ],

                    # SEGMENT TABLE
                    "dw_tableau_marketing_brandsegment_raw": [
                                                                "accountid",
                                                                "account_name",
                                                                "institution_name",
                                                                "npispecialty",
                                                                "segment_product",
                                                                "cluster_name",
                                                                "rundate"
                                                            ]
                }

### Finding all the tables that ended with RAW

Uploaded raw data should be suffixed by _raw

In [0]:
tables_df = spark.sql("""
SELECT table_catalog, table_schema, table_name
FROM system.information_schema.tables
WHERE table_name LIKE '%_raw'
""")

In [0]:
tables_df.show()

### Code to pick the required columns

In [0]:
for row in tables_df.toLocalIterator():

    catalog = row["table_catalog"]
    schema = row["table_schema"]
    raw_table = row["table_name"]

    columns = column_config.get(raw_table)

    if columns is None:
        print(f"Skipping {raw_table} (no column config)")
        continue

    # Source table
    raw_table_full = f"`{catalog}`.{schema}.{raw_table}"

    # Remove suffixes
    clean_schema = schema.replace("_test", "")
    clean_table = raw_table.replace("_raw", "")

    # Destination table
    clean_table_full = f"`{catalog}`.{clean_schema}.{clean_table}"

    try:

        df = spark.table(raw_table_full)

        # Case-insensitive column mapping
        actual_cols = {c.lower(): c for c in df.columns}

        valid_columns = []
        missing_columns = []

        for c in columns:
            if c.lower() in actual_cols:
                valid_columns.append(actual_cols[c.lower()])
            else:
                missing_columns.append(c)

        if missing_columns:
            print(f"Missing columns in {raw_table_full}: {missing_columns}")

        if not valid_columns:
            print(f"No valid columns found for {raw_table_full}")
            continue

        (
            df.select(valid_columns)
            .write
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(clean_table_full)
        )

        print(f"SUCCESS → {clean_table_full}")

    except Exception as e:

        print(f"FAILED → {raw_table_full}")
        print(e)

In [0]:
for raw_table, expected_columns in column_config.items():

    clean_table = raw_table.replace("_raw","")

    tables = spark.sql(f"""
    SELECT table_catalog, table_schema
    FROM system.information_schema.tables
    WHERE table_name = '{clean_table}'
    AND table_schema LIKE '%_test'
    """)

    for row in tables.toLocalIterator():

        catalog = row["table_catalog"]
        schema = row["table_schema"]

        table_full = f"`{catalog}`.{schema}.{clean_table}"

        df = spark.table(table_full)

        actual_column_count = len(df.columns)
        expected_column_count = len(expected_columns)

        if actual_column_count == expected_column_count:
            print(f"✅ QC PASS → {table_full}")
        else:
            print(f"❌ QC FAIL → {table_full}")
            print(f"Expected columns: {expected_column_count}")
            print(f"Actual columns: {actual_column_count}")